In [ ]:
%pip -q install awscli

In [ ]:
%aws s3 ls s3://openfold/ --no-sign-request

                           PRE ablation_checkpoints/
                           PRE alignment_databases/
                           PRE alignment_db/
                           PRE benchmarking_data/
                           PRE converted_checkpoints/
                           PRE data_caches/
                           PRE openfold3_params/
                           PRE openfold_params/
                           PRE openfold_soloseq_params/
                           PRE pdb/
                           PRE soloseq_embeddings/
                           PRE staging/
                           PRE uniclust30/
                           PRE uniclust30_unfiltered/
2022-06-17 03:35:44      18657 LICENSE
2022-08-28 21:57:09    4524064 duplicate_pdb_chains.txt
2024-01-30 15:58:40 55588453165 pdb_mmcif.zip


## Bajar 50 protenias

In [ ]:
%mkdir -p /content/af_subset/msas
%mkdir -p /content/af_subset/jsons
%mkdir -p /content/af_subset/meta

In [ ]:
%aws s3 cp s3://openfold/benchmarking_data/input_jsons_converted/wo_templates/fb_protein.json \
  /content/af_subset/jsons/ --no-sign-request

download: s3://openfold/benchmarking_data/input_jsons_converted/wo_templates/fb_protein.json to af_subset/jsons/fb_protein.json


In [63]:
import json
from pathlib import Path

json_path = Path("/content/af_subset/jsons/fb_protein.json")

with open(json_path, "r") as f:
    data = json.load(f)

targets = []
for qname, q in data["queries"].items():
    for chain in q["chains"]:
        for cid in chain["chain_ids"]:
            if isinstance(cid, str) and len(cid) == 1 and cid.isalpha():
                targets.append(f"{qname.lower()}_{cid}")
                break
        break

targets = sorted(set(targets))
print("n targets:", len(targets))
print(targets[:20])

with open("/content/af_subset/fb_targets_50.txt", "w") as f:
    for t in targets[:50]:
        f.write(t + "\n")

n targets: 239
['7qrj_A', '7qrr_A', '7quv_A', '7qwe_A', '7spq_A', '7th0_A', '7txy_A', '7uq2_A', '7vub_A', '7wr3_A', '7x36_A', '7x80_A', '7xfr_A', '7xft_A', '7xl7_A', '7xn2_A', '7xpi_A', '7xpt_A', '7xrb_A', '7xvq_A']


In [ ]:
%%bash
mkdir -p /content/af_subset/foldbench_msas

while read t; do
  echo "Downloading $t"
  aws s3 cp "s3://openfold/benchmarking_data/msas/foldbench_msas/$t/" \
            "/content/af_subset/foldbench_msas/$t/" \
            --recursive --no-sign-request
done < /content/af_subset/fb_targets_50.txt

In [ ]:
%aws s3 ls s3://openfold/benchmarking_data/reference_structures/ --recursive --no-sign-request | grep -Ei '7QRJ|7QRR|7QUV|7SPQ|7WR3' | head -n 50

2025-11-26 05:15:10     793225 benchmarking_data/reference_structures/foldbench_protein/7qrj-assembly1_68.cif
2025-11-26 05:15:10     272054 benchmarking_data/reference_structures/foldbench_protein/7qrr-assembly1_67.cif
2025-11-26 05:15:11     164863 benchmarking_data/reference_structures/foldbench_protein/7quv-assembly1_70.cif
2025-11-26 05:15:11     146719 benchmarking_data/reference_structures/foldbench_protein/7quv-assembly1_71.cif
2025-11-26 05:15:11    1138414 benchmarking_data/reference_structures/foldbench_protein/7spq-assembly1_142.cif
2025-11-26 05:15:11     864856 benchmarking_data/reference_structures/foldbench_protein/7wr3-assembly1_167.cif


In [ ]:
%%bash
mkdir -p /content/af_subset/reference_structures

while read t; do
  pdb=$(echo "$t" | cut -d'_' -f1 | tr '[:upper:]' '[:lower:]')
  echo "Searching structure for $pdb"

  file=$(aws s3 ls s3://openfold/benchmarking_data/reference_structures/foldbench_protein/ --no-sign-request \
    | awk '{print $4}' \
    | grep -E "^${pdb}-assembly1_.*\.cif$" \
    | head -n 1)

  if [ -n "$file" ]; then
    echo "Downloading $file"
    aws s3 cp "s3://openfold/benchmarking_data/reference_structures/foldbench_protein/$file" \
              "/content/af_subset/reference_structures/$file" \
              --no-sign-request
  else
    echo "NO STRUCTURE FOUND FOR $pdb"
  fi
done < /content/af_subset/fb_targets_50.txt

In [ ]:
%ls /content/af_subset/reference_structures | head
%ls /content/af_subset/reference_structures | wc -l

7qrj-assembly1_68.cif
7qrr-assembly1_67.cif
7quv-assembly1_70.cif
7qwe-assembly1_206.cif
7spq-assembly1_142.cif
7th0-assembly1_130.cif
7txy-assembly1_58.cif
7uq2-assembly1_113.cif
7vub-assembly1_178.cif
7wr3-assembly1_167.cif
50


## Crear un dataset de Referencia

In [68]:
import json
from pathlib import Path
import pandas as pd

json_path = Path("/content/af_subset/jsons/fb_protein.json")
msa_root = Path("/content/af_subset/foldbench_msas")
cif_root = Path("/content/af_subset/reference_structures")

with open(json_path, "r") as f:
    data = json.load(f)

rows = []

for qname, q in data["queries"].items():
    chain = q["chains"][0]
    chain_ids = chain["chain_ids"]
    sequence = chain["sequence"]

    # elegimos la primera cadena alfabética simple, como hicimos al descargar MSAs
    chosen_chain = None
    for cid in chain_ids:
        if isinstance(cid, str) and len(cid) == 1 and cid.isalpha():
            chosen_chain = cid
            break

    if chosen_chain is None:
        continue

    msa_dir_name = f"{qname.lower()}_{chosen_chain}"
    msa_dir = msa_root / msa_dir_name

    cif_candidates = list(cif_root.glob(f"{qname.lower()}-assembly1_*.cif"))
    cif_file = cif_candidates[0] if len(cif_candidates) > 0 else None

    rows.append({
        "query_name": qname,
        "chain_id": chosen_chain,
        "msa_dir_name": msa_dir_name,
        "msa_exists": msa_dir.exists(),
        "msa_dir": str(msa_dir),
        "cif_exists": cif_file is not None,
        "cif_file": str(cif_file) if cif_file is not None else None,
        "seq_len": len(sequence),
        "sequence": sequence})

df_map = pd.DataFrame(rows)
df_map = df_map.dropna(subset=['cif_file'])
print("\nN total:", len(df_map))
print("MSA disponibles:", df_map["msa_exists"].sum())
print("CIF disponibles:", df_map["cif_exists"].sum())
print("Ambos disponibles:", (df_map["msa_exists"] & df_map["cif_exists"]).sum())


N total: 50
MSA disponibles: 50
CIF disponibles: 50
Ambos disponibles: 50


---

In [ ]:
import sys, os

root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_path not in sys.path:
    sys.path.append(root_path)

from data.dataloaders import * 
from data.collate_proteins import *

In [115]:
dataset = FoldbenchProteinDataset(
    json_path="/content/af_subset/jsons/fb_protein.json",
    msa_root="/content/af_subset/foldbench_msas",
    cif_root="/content/af_subset/reference_structures",
    max_msa_seqs=128,
    min_identity=0.85,
    verbose=True,)

print("N ejemplos finales:", len(dataset))
print("Primeros descartados:", dataset.dropped[:10])

sample = dataset[0]
for k, v in sample.items():
    if torch.is_tensor(v):
        print(k, v.shape, v.dtype)
    else:
        print(k, type(v), v)

Dataset valid examples: 39
Dropped examples: 200
  query_name msa_chain_id matched_chain_id  match_identity
0       7QRJ            A                A        1.000000
1       7QRR            A                F        1.000000
2       7QUV            A                B        1.000000
3       7SPQ            A                A        1.000000
4       7TH0            A                A        0.953488
N ejemplos finales: 39
Primeros descartados: [('7QWE', 'no_msa'), ('7XPT', 'no_msa'), ('7XVQ', 'no_msa'), ('7Y37', 'no_chain_match'), ('7YLS', 'no_msa'), ('7YUL', 'no_msa'), ('7ZOP', 'no_msa'), ('8A51', 'no_msa'), ('8AMY', 'no_msa'), ('8AN5', 'no_msa')]
id <class 'str'> 7QRJ
msa_chain_id <class 'str'> A
matched_chain_id <class 'str'> A
match_identity torch.Size([]) torch.float32
sequence_str <class 'str'> MSISSLLEKNIYNVHNKSNTLTNVPANPTGNTNTVWSNSNFTPPHLMYGASDITQAIGNISLTTGSFSLSLSGPWASPLVQNVAYTKINNLVNLTFPPFQANATSSAVINSAIGALPADLRPTTNIQVDFEIFVIDDGNRPVNPGLITLLSNGQIVVYKDNNLGQFTTGIGGSGFNPFSIT
seq_to

## 1) Qué significa cada dimensión en un ejemplo individual del dataset

Antes del `DataLoader`, cada item representa **una sola proteína/cadena**.

- `id`: identificador del ejemplo, por ejemplo `7QRJ`.
- `msa_chain_id`: cadena usada para buscar el MSA en los archivos de entrada.
- `matched_chain_id`: cadena del `.cif` que realmente hizo match con la secuencia target.
- `match_identity`: escalar que mide qué tan bien coincidió la secuencia target con la cadena estructural encontrada.

### Tensores principales

- `sequence_str`: secuencia como texto, una letra por residuo.
- `seq_tokens [L]`: secuencia tokenizada.  
  Aquí `L` es la longitud de la proteína. Cada posición corresponde a un residuo.

- `msa_tokens [N_msa, L]`: MSA tokenizado.  
  - `N_msa`: número de secuencias homólogas usadas en el alineamiento.
  - `L`: longitud alineada de la proteína.  
  Cada fila es una secuencia del MSA y cada columna una posición del residuo.

- `msa_mask [N_msa, L]`: máscara del MSA.  
  Indica qué posiciones son válidas y cuáles son gaps o padding.

### Coordenadas estructurales

- `coords_n [L, 3]`: coordenadas 3D del átomo **N** para cada residuo.
- `coords_ca [L, 3]`: coordenadas 3D del átomo **Cα** para cada residuo.
- `coords_c [L, 3]`: coordenadas 3D del átomo **C** para cada residuo.

En todos estos casos:
- la primera dimensión recorre residuos,
- la segunda dimensión tiene las coordenadas espaciales `(x, y, z)`.

### Información geométrica y máscaras

- `dist_map [L, L]`: matriz de distancias entre residuos, usualmente usando Cα.  
  `dist_map[i, j]` es la distancia entre los residuos `i` y `j`.

- `valid_res_mask [L]`: máscara por residuo.  
  Vale `1` si el residuo tiene `CA` válido y `0` si no.

- `valid_backbone_mask [L]`: máscara más estricta.  
  Vale `1` si el residuo tiene `N`, `CA` y `C`; `0` si falta alguno.

In [116]:
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_proteins)

batch = next(iter(loader))

for k, v in batch.items():
    if torch.is_tensor(v):
        print(k, v.shape, v.dtype)
    else:
        print(k, type(v))

id <class 'list'>
msa_chain_id <class 'list'>
matched_chain_id <class 'list'>
match_identity torch.Size([2]) torch.float32
sequence_str <class 'list'>
seq_tokens torch.Size([2, 250]) torch.int64
seq_mask torch.Size([2, 250]) torch.float32
msa_tokens torch.Size([2, 128, 250]) torch.int64
msa_mask torch.Size([2, 128, 250]) torch.float32
coords_n torch.Size([2, 250, 3]) torch.float32
coords_ca torch.Size([2, 250, 3]) torch.float32
coords_c torch.Size([2, 250, 3]) torch.float32
dist_map torch.Size([2, 250, 250]) torch.float32
valid_res_mask torch.Size([2, 250]) torch.float32
valid_backbone_mask torch.Size([2, 250]) torch.float32
pair_mask torch.Size([2, 250, 250]) torch.float32
backbone_pair_mask torch.Size([2, 250, 250]) torch.float32
torsion_true torch.Size([2, 250, 3, 2]) torch.float32
torsion_mask torch.Size([2, 250, 3]) torch.float32


## 2) Qué significa cada dimensión después del DataLoader

Después del `DataLoader`, ya no tienes una sola proteína sino un **batch** de tamaño `B`.

Si el batch tiene tamaño 2, entonces aparecen shapes como:

- `seq_tokens [B, L_max]`
- `msa_tokens [B, N_msa, L_max]`
- `coords_ca [B, L_max, 3]`

donde:

- `B`: número de proteínas del batch.
- `L_max`: longitud máxima entre las proteínas del batch.
- `N_msa`: número de secuencias del MSA por proteína.

### Secuencia y MSA en batch

- `seq_tokens [B, L_max]`: secuencias tokenizadas batcheadas.  
  Cada fila es una proteína y cada columna una posición del residuo.

- `seq_mask [B, L_max]`: máscara de secuencia.  
  Vale `1` donde hay residuo real y `0` donde el `collate` agregó padding.

- `msa_tokens [B, N_msa, L_max]`: MSA batcheado.  
  - eje 0: proteína del batch
  - eje 1: fila del MSA
  - eje 2: posición del residuo

- `msa_mask [B, N_msa, L_max]`: máscara del MSA con la misma estructura.

### Coordenadas en batch

- `coords_n [B, L_max, 3]`
- `coords_ca [B, L_max, 3]`
- `coords_c [B, L_max, 3]`

Aquí:
- eje 0 = proteína,
- eje 1 = residuo,
- eje 2 = coordenadas `(x, y, z)`.

### Distancias y máscaras de pares

- `dist_map [B, L_max, L_max]`: mapa de distancias por proteína.  
  `dist_map[b, i, j]` es la distancia entre los residuos `i` y `j` de la proteína `b`.

- `valid_res_mask [B, L_max]`: máscara de residuos con `CA` válido.
- `valid_backbone_mask [B, L_max]`: máscara de residuos con backbone completo (`N`, `CA`, `C`).

- `pair_mask [B, L_max, L_max]`: máscara sobre pares de residuos.  
  Vale `1` si ambos residuos del par son válidos.  
  Sirve para módulos sobre representaciones de pares `z[i, j]`.

- `backbone_pair_mask [B, L_max, L_max]`: versión más estricta del `pair_mask`, usando solo residuos con backbone completo.

### Idea general

Tu pipeline organiza la información en tres niveles:

1. **Secuencia**  
   `seq_tokens`, `seq_mask`

2. **MSA**  
   `msa_tokens`, `msa_mask`

3. **Estructura / geometría**  
   `coords_*`, `dist_map`, `valid_res_mask`, `valid_backbone_mask`, `pair_mask`, `backbone_pair_mask`

Eso encaja muy bien con AlphaFold, porque el modelo combina:
- información de secuencia,
- información evolutiva del MSA,
- y supervisión estructural.

## Train Model

In [ ]:
import torch
from training.train_alphafold2 import *
from model.alphafold2 import *
from model.alphafold2_full_loss import *

# ============================================================
# Seed + device
# ============================================================
seed_everything(42, deterministic=False)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# ============================================================
# Ideal local backbone geometry
#    orden: N, CA, C, O
# ============================================================
ideal_backbone_local = torch.tensor([
    [-1.458,  0.000,  0.000],   # N
    [ 0.000,  0.000,  0.000],   # CA
    [ 0.547,  1.426,  0.000],   # C
    [ 0.224,  2.617,  0.000],   # O
], dtype=torch.float32)

# ============================================================
# Model
# ============================================================
n_tokens = max(AA_VOCAB.values()) + 1   # aquí 27 con vocab actual
pad_idx = AA_VOCAB["-"]

model = AlphaFold2(
    n_tokens=n_tokens,
    c_m=256,
    c_z=128,
    c_s=256,
    max_relpos=32,
    pad_idx=pad_idx,
    num_evoformer_blocks=2,   # para PoC; luego puedes subir
    num_structure_blocks=4,   # para PoC; luego puedes subir
    dist_bins=64,
    plddt_bins=50,
    n_torsions=3, transition_expansion_evoformer=2 ,
    transition_expansion_structure = 2,
    use_block_specific_params = False).to(device)

# ============================================================
# Loss
# ============================================================
criterion = AlphaFoldLoss(
    fape_length_scale=10.0,
    fape_clamp_distance=10.0,
    dist_num_bins=64,
    dist_min_bin=2.0,
    dist_max_bin=22.0,
    plddt_num_bins=50,
    plddt_inclusion_radius=15.0,
    w_fape=0.5,
    w_dist=0.3,
    w_plddt=0.01,
    w_torsion=0.01,)

# ============================================================
# Optimizer + Scheduler
#    total_steps debe ser en optimizer steps, no epochs
# ============================================================
epochs = 20
grad_accum_steps = 1

steps_per_epoch = (len(loader) + grad_accum_steps - 1) // grad_accum_steps
total_steps = epochs * steps_per_epoch
warmup_steps = max(10, int(0.05 * total_steps))   # 5% de warmup como heurística inicial

optimizer, scheduler = build_optimizer_and_scheduler(
    model=model,
    lr=1e-4,
    weight_decay=1e-4,
    betas=(0.9, 0.95),
    eps=1e-8,
    total_steps=total_steps,
    warmup_steps=warmup_steps,
    min_lr=1e-6)

# ============================================================
# EMA
# ============================================================
ema = EMA(
    model=model,
    decay=0.999,
    device="cpu",          # "cpu" para ahorrar VRAM
    use_num_updates=True,)

# ============================================================
# AMP config
# ============================================================
amp_cfg = build_amp_config(
    device=device,
    amp_enabled=True,
    amp_dtype="bf16")

scaler = amp_cfg["scaler"]

print("AMP cfg:", amp_cfg)


In [ ]:
# ============================================================
# Train
# ============================================================
result = train_alphafold2(
    model=model,
    train_loader=loader,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    ema=ema,
    scaler=scaler,
    device=device,
    epochs=epochs,
    start_epoch=0,
    global_step=0,
    amp_enabled=amp_cfg["amp_enabled"],
    amp_dtype=amp_cfg["amp_dtype_requested"],
    grad_clip=1.0,
    grad_accum_steps=grad_accum_steps,
    log_every=5,
    log_grad_norm=True,
    log_mem=True,
    monitor_name="loss",          #estructurales: "rmsd_logged", "tm_score_logged", "gdt_ts_logged"
    monitor_mode="min",
    best_metric=None,
    ckpt_dir="checkpoints_af2",
    run_name="af2_poc",
    save_every=1,
    save_last=True,
    resume_path=None,
    ideal_backbone_local=ideal_backbone_local.to(device),
    drive_ckpt_dir="/content/drive/MyDrive/af2_ckpts" if is_colab() else None,
    copy_fixed_to_drive=True,
    num_recycles=0,
    stochastic_recycling=False,
    fixed_drive_name="latest_af2.pt",
    config={
        "amp_dtype": amp_cfg["amp_dtype_requested"],
        "grad_clip": 1.0,
        "epochs": epochs,
        "lr": 1e-4,
        "weight_decay": 1e-4,
        "total_steps": total_steps,
        "warmup_steps": warmup_steps,
        "num_evoformer_blocks": 2,
        "num_structure_blocks": 4,
    },
)

print(result)

---

## Flujo completo del entrenamiento: qué entra, qué produce el modelo y qué consume la loss

En esta etapa, el `DataLoader` entrega un batch con información de **secuencia**, **MSA** y **estructura real**.

### 1) Qué entra realmente al modelo

En `train_one_epoch`, al `model(...)` se le pasan directamente:

- `seq_tokens [B, L]`
- `msa_tokens [B, N_msa, L]`
- `seq_mask [B, L]`
- `msa_mask [B, N_msa, L]`
- `ideal_backbone_local` (constante geométrica para reconstruir backbone)

Es decir, el modelo **no recibe directamente las coordenadas reales** como input del forward.  
Las coordenadas verdaderas se reservan para la supervisión en la loss y las métricas.

### 2) Qué hace AlphaFold dentro del forward

El flujo interno es:

**(a) InputEmbedder**  
A partir de `seq_tokens` y `msa_tokens`, construye:

- `m`: representación inicial del MSA
- `z`: representación inicial de pares residuo-residuo

**(b) Evoformer**  
El Evoformer toma `m` y `z` y los refina mediante atención sobre MSA, outer product mean y bloques triangulares.  
Después del Evoformer, salen:

- `m` enriquecido
- `z` enriquecido

**(c) SingleProjection**  
A partir del MSA refinado, se construye:

- `s`: representación single por residuo

**(d) StructureModule**  
El structure module toma `s` y `z`, y va actualizando marcos rígidos residuo por residuo.  
De ahí salen:

- `R [B, L, 3, 3]`: rotaciones por residuo
- `t [B, L, 3]`: traslaciones por residuo

A partir de `R`, `t` e `ideal_backbone_local`, el modelo reconstruye además:

- `backbone_coords [B, L, A, 3]` con los átomos backbone predichos

**(e) Heads auxiliares**  
Sobre `s` y `z` el modelo también produce:

- `torsions`
- `plddt_logits`
- `plddt`
- `distogram_logits`

Por eso el output final del modelo es:

```python
{
    "m": m,
    "z": z,
    "s": s,
    "R": R,
    "t": t,
    "backbone_coords": backbone_coords,
    "torsions": torsions,
    "plddt_logits": plddt_logits,
    "plddt": plddt,
    "distogram_logits": distogram_logits
}

## Después del output del modelo: qué pasa en la loss, las métricas y qué partes del batch sí se usan

Una vez termina el `forward`, el modelo ya produjo sus representaciones internas refinadas (`m`, `z`, `s`), sus variables geométricas (`R`, `t`, `backbone_coords`) y sus heads auxiliares (`torsions`, `plddt_logits`, `plddt`, `distogram_logits`). En ese momento, esas predicciones no se usan solas: pasan inmediatamente a la **loss**, donde se comparan contra la estructura real que venía en el batch del `DataLoader`.

### 1) Qué consume la loss

La loss toma del **output del modelo**:

- `R`: rotaciones por residuo
- `t`: traslaciones por residuo
- `backbone_coords`: backbone predicho
- `distogram_logits`: predicción de distancias entre residuos
- `plddt_logits`: predicción de confianza local
- `torsions`: predicción de ángulos torsionales, si existe supervisión para eso

Y del **batch real** toma:

- `coords_n`: coordenadas reales del átomo N
- `coords_ca`: coordenadas reales del átomo CA
- `coords_c`: coordenadas reales del átomo C
- `valid_res_mask`: máscara de residuos con CA válido
- `valid_backbone_mask`: máscara de residuos con backbone completo (N, CA, C)
- `torsion_true`
- `torsion_mask`

### 2) Qué calcula la loss

Con eso, tu `AlphaFoldLoss` construye varias pérdidas parciales:

- **FAPE loss**: compara la geometría predicha frente a la geometría real usando marcos locales; aquí importan especialmente `R`, `t` y las coordenadas backbone verdaderas.
- **Distogram loss**: compara los logits de distancia predichos con las distancias reales entre residuos.
- **pLDDT loss**: supervisa si el modelo está asignando bien la confianza local por residuo.
- **Torsion loss**: solo si existen torsiones verdaderas en el batch.

Al final, la loss devuelve un diccionario con algo como:

- `loss`
- `fape_loss`
- `dist_loss`
- `plddt_loss`
- `torsion_loss`

La que se usa para `backward()` es la combinada: `loss`.

### 3) Qué usan las métricas estructurales

Después de la loss, en el bloque de métricas, no se usan todas las salidas del modelo. Para `RMSD`, `TM-score` y `GDT-TS`, en el loop actual se toma únicamente la geometría de los **Cα predichos** y se compara contra los **Cα reales**.

Concretamente:

- predicción: normalmente el CA extraído de `backbone_coords`
- verdad: `coords_ca`
- máscara: `valid_res_mask`

Luego se hace alineación Kabsch y sobre esa alineación se calculan:

- **RMSD**
- **TM-score**
- **GDT-TS**

Estas métricas no participan en el gradiente; son solo monitoreo.

### 4) Qué partes del batch sí entran en cada etapa

**Entran al modelo**
- `seq_tokens`
- `msa_tokens`
- `seq_mask`
- `msa_mask`

**Entran a la loss**
- `coords_n`
- `coords_ca`
- `coords_c`
- `valid_res_mask`
- `valid_backbone_mask`

**Entran a las métricas**
- `coords_ca`
- `valid_res_mask`



### 5) Resumen conceptual del pipeline completo

El flujo completo de entrenamiento queda así:

1. **El DataLoader** entrega secuencia, MSA y estructura real.
2. **El modelo** usa solo secuencia + MSA para producir representaciones y geometría predicha.
3. **La loss** compara esa geometría predicha contra la estructura real del batch.
4. **Las métricas** comparan CA predicho vs CA real para monitorear calidad estructural.
5. **El backward** se hace únicamente con la `loss` total, no con RMSD/TM/GDT.

En otras palabras: el modelo aprende a predecir estructura a partir de secuencia y MSA, y la supervisión real entra después, cuando comparas su output con las coordenadas verdaderas.